In [ ]:
import numpy as np
from numpy.linalg import solve, norm
import pandas as pd
from itertools import accumulate, groupby
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
dim = 10

# Steps

In [ ]:
def proj_simplex(vec, rad=1):
    'https://stanford.edu/~jduchi/projects/DuchiShSiCh08.pdf'
    muu = np.sort(vec)[::-1]
    cummeans = 1 / np.arange(1, len(vec) + 1) * (np.cumsum(muu) - rad)
    rho = max(np.where(muu > cummeans)[0])
    proj = np.maximum(vec - cummeans[rho], 0)
    return proj

In [ ]:
def zero_grad(trans, pi0):
    mat = np.block([[np.eye(len(trans)), pi0], [pi0.T, 0]])
    vec = np.concatenate([trans, pi0.T])
    res = solve(mat, vec)[:-1, :]
    return res

In [ ]:
trans0 = np.abs(np.random.randn(dim, dim))
trans0 /= trans0.sum(axis=1).reshape(-1, 1)
trans0, trans0.sum(axis=1)

In [ ]:
pi0 = np.abs(np.random.randn(dim, 1))
pi0 /= pi0.sum()
pi0

In [ ]:
zero = zero_grad(trans0, pi0)
zero

In [ ]:
zero.T @ pi0 - pi0, norm(zero - trans0)

In [ ]:
def normalization(zero):
    return np.array([proj_simplex(row) for row in zero])

In [ ]:
normalization(zero)

# Iterations

In [ ]:
def proj_grad(trans0, pi0, err_max=1e-6, ite_max=1_000):
    err = err_max + 1
    ite = 0
    trans = trans0.copy()
    errors = list()
    while err > err_max and ite < ite_max:
        trans_new = normalization(zero_grad(trans, pi0))
        err = norm(trans - trans_new)
        errors.append(err)
        trans = trans_new.copy()
        ite += 1
    return trans, errors

In [ ]:
trans, errors = proj_grad(trans0, pi0)
pd.Series(errors).apply(np.log).plot()
trans
trans.T @ pi0 - pi0, trans.sum(axis=1), norm(trans0 - trans)

# Repartition functions

In [ ]:
sample = np.random.randn(10000)

def get_cdf(sample):
    return sorted(sample), np.linspace(0, 1, len(sample))

plt.plot(*get_cdf(sample))

In [ ]:
xxx_0, yyy_0 = get_cdf(np.random.randn(10000))
xxx_1, yyy_1 = get_cdf(np.random.randn(10000))

In [ ]:
supp_min = max(min(xxx_0), min(xxx_1))
supp_max  = min(max(xxx_0), max(xxx_1))
supp_min, supp_max

In [ ]:
def restrict(xxx_0, yyy_0, x_min, x_max):
    return zip(*[
        (xxx, yyy) for xxx, yyy in zip(xxx_0, yyy_0)
        if x_min <= xxx <= x_max])

In [ ]:
supp_union = sorted(
    ele for ele in set(xxx_0) | set(xxx_1)
    if supp_min <= ele <= supp_max)
xxx_0, yyy_0 = restrict(xxx_0, yyy_0, supp_min, supp_max)
xxx_1, yyy_1 = restrict(xxx_1, yyy_1, supp_min, supp_max)

In [ ]:
plt.plot(xxx_0, yyy_0)

In [ ]:
plt.plot(xxx_1, yyy_1)

In [ ]:
y_int_0 = np.interp(supp_union, xxx_0, yyy_0)
y_int_1 = np.interp(supp_union, xxx_1, yyy_1)

In [ ]:
plt.plot(supp_union, y_int_0)

In [ ]:
max(abs(y_int_0 - y_int_1))

In [ ]:
def dist_cdfs(xxx_0, yyy_0, xxx_1, yyy_1):
    supp_min = max(min(xxx_0), min(xxx_1))
    supp_max  = min(max(xxx_0), max(xxx_1))
    xxx_0, yyy_0 = restrict(xxx_0, yyy_0, supp_min, supp_max)
    xxx_1, yyy_1 = restrict(xxx_1, yyy_1, supp_min, supp_max)
    supp_union = sorted(
        ele for ele in set(xxx_0) | set(xxx_1)
        if supp_min <= ele <= supp_max)
    y_int_0 = np.interp(supp_union, xxx_0, yyy_0)
    y_int_1 = np.interp(supp_union, xxx_1, yyy_1)
    return max(abs(y_int_0 - y_int_1))

In [ ]:
xxx_0, yyy_0 = get_cdf(np.random.randn(100000))
xxx_1, yyy_1 = get_cdf(np.random.randn(100000))
dist_cdfs(xxx_0, yyy_0, xxx_1, yyy_1)

# Max DD

In [ ]:
data = pd.Series(1e-4 + 2e-3 * np.random.randn(1000)).add(1).cumprod()
convex_env = data.expanding().max()
data.plot()
convex_env.plot()

In [ ]:
(data / convex_env).plot()

In [ ]:
dds = 1 - data / convex_env
dds = dds[dds > 0]
dds.max()

In [ ]:
dds.hist(bins=100)

In [ ]:
plt.plot(sorted(dds.values))

In [ ]:
test = np.random.randn(10000)
plt.plot(sorted(test), np.linspace(0, 1, len(test)))

In [ ]:
def quantile_dds(ts_, percentiles=None):
    percentiles = [0.95, 0.5] if percentiles is None else percentiles
    convex_env = list(accumulate(ts_, max))
    dds = 1 - ts_ / convex_env
    dds_inds = [
        list(zip(*vals))[0] for ind, vals in
        groupby(enumerate(dds > 0), lambda cpl: cpl[1]) if ind]
    dds_data = [
        [0] + [dds[ind] for ind in sublist] + [0]
        for sublist in dds_inds[:-1]]
    dds_data = pd.DataFrame(dds_data).fillna(0)
    curves = {
        perc: dds_data.quantile(perc, axis=0) for perc in percentiles}
    return curves

In [ ]:
data = pd.Series(2e-3 * np.random.randn(1000000)).add(1).cumprod()
for perc, curve in quantile_dds(data).items():
    print(perc)
    plt.figure()
    plt.plot(curve.iloc[:500])
    plt.show()

In [ ]:
convex_env = list(accumulate(data, max))
dds = 1 - data / convex_env
plt.plot(dds)

In [ ]:
for a, b in groupby(enumerate(dds > 0), lambda x: x[1]):
    print(a, list(zip(*b))[0])

In [ ]:
dds_inds = [
    list(zip(*vals))[0] for ind, vals in
    groupby(enumerate(dds > 0), lambda cpl: cpl[1])
    if ind]
dds_inds

In [ ]:
dds_data = pd.DataFrame([
    [0] + [dds[ind] for ind in sublist] + [0]
    for sublist in dds_inds[:-1]])
dds_data = dds_data.fillna(0).quantile(0.01, axis=0)
dds_data.iloc[:10].plot()

In [ ]:
def get_cdf_dds(data):
    convex_env = list(accumulate(data, max))
    dds = 1 - data / convex_env
    dds = dds[dds > 0]
    dds.max()
    return sorted(dds), np.linspace(0, 1, len(dds))

In [ ]:
params = {'mean': 1e-4, 'var': 2e-3}
data = params['mean'] + params['var'] * np.random.randn(1000)
data = (data + 1).cumprod()
xxx1, yyy1 = get_cdf_dds(data)

In [ ]:
plt.plot(xxx1, yyy1)